# 厦门站 1985 年 CNN 风暴潮重建：模型训练与验证

这个 notebook 接在 `Data_preprocessing.ipynb` 后面运行。

前一个 notebook 已经生成了 CNN 可用数据：

- `X_train.npy`：训练集输入，形状应为 `(N_train, 48, 40, 40)`；
- `y_train.npy`：训练集标签，已经标准化；
- `X_val.npy`：验证集输入，形状应为 `(N_val, 48, 40, 40)`；
- `y_val.npy`：验证集标签，已经标准化；
- `dates_val.npy`：验证集日期；
- `y_original.npy` / `y_scaler.json`：用于把预测值还原到原始 storm surge 单位。

这里不再做数据预处理，只做 CNN 训练、验证、画图和保存结果。

In [ ]:
# ============================================================
# 0. 导入模型训练需要用到的库
# ============================================================

import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-ticks")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# ============================================================
# 1. 设置随机种子，让训练结果尽量可复现
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("随机种子:", SEED)

In [ ]:
# ============================================================
# 2. 设置输入数据目录和模型输出目录
# ============================================================

# 这里对应 Data_preprocessing.ipynb 中的 OUT_DIR。
# 如果你在实验室电脑上改了预处理输出目录，只需要改这里。
DATA_DIR = Path(r"E:\AAAqian\code\storm_surge\data\processed_xiamen_1985")

# 训练结果保存位置。
# runs/ 已经写入 .gitignore，不会上传 GitHub。
RUN_DIR = Path(r"E:\AAAqian\code\storm_surge\runs\xiamen_1985")
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("数据目录:", DATA_DIR)
print("输出目录:", RUN_DIR)

# 训练前检查关键文件是否存在。
required_files = [
    "X_train.npy", "y_train.npy",
    "X_val.npy", "y_val.npy",
    "dates_train.npy", "dates_val.npy",
    "y_original.npy", "dates_all.npy",
]

missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        "预处理结果不完整，请先完整运行 Data_preprocessing.ipynb。缺少文件: "
        + ", ".join(missing_files)
    )

print("预处理文件检查通过。")

In [ ]:
# ============================================================
# 3. 读取训练集和验证集
# ============================================================

# X 可能比较大，所以用 mmap_mode="r"。
# 这样不会一次性把全部 X 加载进内存，而是训练时按 batch 读取。
X_train = np.load(DATA_DIR / "X_train.npy", mmap_mode="r")
y_train = np.load(DATA_DIR / "y_train.npy")

X_val = np.load(DATA_DIR / "X_val.npy", mmap_mode="r")
y_val = np.load(DATA_DIR / "y_val.npy")

dates_train = np.load(DATA_DIR / "dates_train.npy", allow_pickle=True)
dates_val = np.load(DATA_DIR / "dates_val.npy", allow_pickle=True)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("训练日期范围:", dates_train[0], "到", dates_train[-1])
print("验证日期范围:", dates_val[0], "到", dates_val[-1])

In [ ]:
# ============================================================
# 4. 检查 CNN 输入形状是否符合论文逻辑
# ============================================================

# 每个样本应该是：
# 2 天 × 每天 8 个 3小时时间片 × 3 个变量 = 48 个通道
# 每个通道是 40 × 40 网格

assert X_train.ndim == 4, "X_train 应该是 4 维: (N, 48, 40, 40)"
assert X_val.ndim == 4, "X_val 应该是 4 维: (N, 48, 40, 40)"
assert X_train.shape[1:] == (48, 40, 40), f"X_train shape 不对: {X_train.shape}"
assert X_val.shape[1:] == (48, 40, 40), f"X_val shape 不对: {X_val.shape}"
assert len(X_train) == len(y_train), "训练集 X/y 数量不一致"
assert len(X_val) == len(y_val), "验证集 X/y 数量不一致"

# 抽查第一个样本是否有 NaN 或 Inf。
sample0 = np.asarray(X_train[0])
print("单个样本 shape:", sample0.shape)
print("第一个样本是否有 NaN:", np.isnan(sample0).any())
print("第一个样本是否有 Inf:", np.isinf(sample0).any())
print("y_train 前5个:", y_train[:5])
print("y_val 前5个:", y_val[:5])

In [ ]:
# ============================================================
# 5. 读取 y 标准化参数，用于把预测值还原到原始单位
# ============================================================

# 训练时 y_train/y_val 是标准化后的值。
# 画图、计算 RMSE/MAE 时，我们更关心原始单位下的 storm surge。

scaler_file = DATA_DIR / "y_scaler.json"

if scaler_file.exists():
    with open(scaler_file, "r", encoding="utf-8") as f:
        y_scaler_info = json.load(f)
    y_mean = float(y_scaler_info["mean"])
    y_std = float(y_scaler_info["std"])
else:
    # 如果旧版预处理 notebook 没保存 y_scaler.json，
    # 就用 y_original 和 y_train/y_val 反推出标准化参数。
    y_original = np.load(DATA_DIR / "y_original.npy").astype("float64")
    y_scaled_all = np.concatenate([y_train, y_val]).astype("float64")
    design = np.column_stack([y_scaled_all, np.ones_like(y_scaled_all)])
    y_std, y_mean = np.linalg.lstsq(design, y_original, rcond=None)[0]

print("y_mean:", y_mean)
print("y_std:", y_std)

def inverse_transform_y(y_scaled):
    """把标准化后的 y 还原为原始 storm surge 单位。"""
    return y_scaled * y_std + y_mean

In [ ]:
# ============================================================
# 6. 定义 PyTorch Dataset
# ============================================================

class StormSurgeDataset(Dataset):
    """
    把 numpy 数组包装成 PyTorch 可以训练的数据集。

    注意：
    X 是 mmap 数组时，__getitem__ 每次只读取一个样本，
    不会一次性把全部 X 放进内存。
    """

    def __init__(self, X, y):
        self.X = X
        self.y = y.astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = np.asarray(self.X[idx], dtype=np.float32)
        y = np.float32(self.y[idx])
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.float32)


train_dataset = StormSurgeDataset(X_train, y_train)
val_dataset = StormSurgeDataset(X_val, y_val)

print("训练集样本数:", len(train_dataset))
print("验证集样本数:", len(val_dataset))

In [ ]:
# ============================================================
# 7. 设置 DataLoader 和训练设备
# ============================================================

# 1985 单年样本较少，batch_size=16 通常足够。
# 如果显存不够，可以改成 8 或 4。
BATCH_SIZE = 16

# Windows 下 num_workers 先设为 0，最稳定。
NUM_WORKERS = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("训练设备:", device)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

In [ ]:
# ============================================================
# 8. 定义 CNN 模型
# ============================================================

class StormSurgeCNN(nn.Module):
    """
    适配 (48, 40, 40) 输入的 CNN。

    为什么输入通道是 48？
    - 前一天 + 当天 = 2 天
    - ERA20C 每 3 小时一个时间片，每天 8 个
    - 两天共 16 个时间片
    - U10、V10、SLP 三个变量
    - 16 × 3 = 48 个通道

    输出是 1 个数：当天 daily maximum storm surge 的标准化值。
    """

    def __init__(self, in_channels=48, dropout=0.30):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 40 -> 20

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 20 -> 10

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 10 -> 5
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 5 * 5, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x.squeeze(-1)


model = StormSurgeCNN(in_channels=48, dropout=0.30).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print("可训练参数量:", num_params)

In [ ]:
# ============================================================
# 9. 设置损失函数、优化器和训练参数
# ============================================================

# 回归任务常用 MSELoss。
loss_fn = nn.MSELoss()

# AdamW 比普通 SGD 更容易先跑通。
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# 如果验证集 loss 长时间不下降，自动降低学习率。
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=8
)

# 1985 单年样本少，先用 100 epoch 跑通流程。
N_EPOCHS = 100

# 早停：验证集 loss 连续多少轮不提升就停止。
EARLY_STOP_PATIENCE = 25

print("学习率:", LEARNING_RATE)
print("训练轮数:", N_EPOCHS)

In [ ]:
# ============================================================
# 10. 定义训练一轮和验证一轮的函数
# ============================================================

def run_one_epoch(model, loader, loss_fn, optimizer=None):
    """
    运行一个 epoch。

    optimizer 不是 None：训练模式，会反向传播并更新参数。
    optimizer 是 None：验证模式，只计算 loss，不更新参数。
    """

    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_count = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            y_pred = model(x_batch)
            loss = loss_fn(y_pred, y_batch)

            if is_train:
                loss.backward()
                optimizer.step()

        batch_size = len(y_batch)
        total_loss += loss.item() * batch_size
        total_count += batch_size

    return total_loss / total_count

In [ ]:
# ============================================================
# 11. 正式训练 CNN
# ============================================================

history = []

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

best_model_path = RUN_DIR / "best_model.pth"

for epoch in range(1, N_EPOCHS + 1):
    train_loss = run_one_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss = run_one_epoch(model, val_loader, loss_fn, optimizer=None)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "lr": current_lr,
    })

    # 如果当前验证 loss 是历史最好，就保存模型。
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        # 注意：这里把 epoch/loss/scaler 都转成纯 Python int/float。
        # 这样 checkpoint 更容易被 PyTorch 新版本安全加载机制识别。
        torch.save({
            "model_state_dict": model.state_dict(),
            "best_epoch": int(best_epoch),
            "best_val_loss": float(best_val_loss),
            "y_mean": float(y_mean),
            "y_std": float(y_std),
        }, best_model_path)
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.6f} | "
            f"val_loss={val_loss:.6f} | "
            f"lr={current_lr:.2e}"
        )

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(f"验证集 loss 连续 {EARLY_STOP_PATIENCE} 轮没有提升，提前停止。")
        break

print("训练完成。")
print("最佳 epoch:", best_epoch)
print("最佳 val_loss:", best_val_loss)
print("最佳模型保存到:", best_model_path)

In [ ]:
# ============================================================
# 12. 保存并绘制训练曲线
# ============================================================

history_df = pd.DataFrame(history)
history_path = RUN_DIR / "training_history.csv"
history_df.to_csv(history_path, index=False, encoding="utf-8-sig")

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Train loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss (standardized y)")
plt.title("Xiamen CNN training curve")
plt.grid(True)
plt.legend()
plt.tight_layout()

loss_fig_path = RUN_DIR / "loss_curve.png"
plt.savefig(loss_fig_path, dpi=200)
plt.show()

print("训练历史保存到:", history_path)
print("loss 曲线保存到:", loss_fig_path)

In [ ]:
# ============================================================
# 13. 加载最佳模型，在验证集上预测
# ============================================================

# PyTorch 2.6 以后 torch.load 默认 weights_only=True。
# 这里的 best_model.pth 是我们自己在上一个 Cell 训练并保存的本地文件，
# 里面除了模型权重，还保存了 best_epoch、best_val_loss、y_mean、y_std 等信息，
# 所以需要显式设置 weights_only=False。
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

pred_scaled_list = []
true_scaled_list = []

with torch.no_grad():
    for x_batch, y_batch in val_loader:
        x_batch = x_batch.to(device)
        pred_scaled = model(x_batch).cpu().numpy()

        pred_scaled_list.append(pred_scaled)
        true_scaled_list.append(y_batch.numpy())

y_val_true_scaled = np.concatenate(true_scaled_list)
y_val_pred_scaled = np.concatenate(pred_scaled_list)

print("标准化预测 shape:", y_val_pred_scaled.shape)
print("标准化真实 shape:", y_val_true_scaled.shape)

In [ ]:
# ============================================================
# 14. 反标准化，并计算验证集指标
# ============================================================

y_val_true = inverse_transform_y(y_val_true_scaled)
y_val_pred = inverse_transform_y(y_val_pred_scaled)

rmse = mean_squared_error(y_val_true, y_val_pred) ** 0.5
mae = mean_absolute_error(y_val_true, y_val_pred)
r2 = r2_score(y_val_true, y_val_pred)
corr = np.corrcoef(y_val_true, y_val_pred)[0, 1]

metrics = {
    "best_epoch": int(best_epoch),
    "best_val_loss_standardized": float(best_val_loss),
    "rmse_original_unit": float(rmse),
    "mae_original_unit": float(mae),
    "r2": float(r2),
    "corr": float(corr),
    "n_train": int(len(train_dataset)),
    "n_val": int(len(val_dataset)),
}

metrics_path = RUN_DIR / "metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

display(pd.DataFrame([metrics]))
print("指标保存到:", metrics_path)

In [ ]:
# ============================================================
# 15. 保存验证集逐日预测结果
# ============================================================

pred_df = pd.DataFrame({
    "date": pd.to_datetime(dates_val).astype(str),
    "observed": y_val_true,
    "predicted": y_val_pred,
})

pred_df["error"] = pred_df["predicted"] - pred_df["observed"]

pred_path = RUN_DIR / "val_predictions.csv"
pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

display(pred_df.head())
print("验证集预测保存到:", pred_path)

In [ ]:
# ============================================================
# 16. 画验证集时间序列：真实值 vs 预测值
# ============================================================

plt.figure(figsize=(12, 4))
plt.plot(pred_df["date"], pred_df["observed"], label="Observed", linewidth=1.2)
plt.plot(pred_df["date"], pred_df["predicted"], label="Predicted", linewidth=1.2)
plt.xticks(rotation=45)
plt.xlabel("Date")
plt.ylabel("Daily maximum storm surge")
plt.title("Xiamen validation: observed vs predicted")
plt.grid(True)
plt.legend()
plt.tight_layout()

timeseries_path = RUN_DIR / "val_timeseries.png"
plt.savefig(timeseries_path, dpi=200)
plt.show()

print("时间序列图保存到:", timeseries_path)

In [ ]:
# ============================================================
# 17. 画散点图：真实值 vs 预测值
# ============================================================

plt.figure(figsize=(5, 5))
plt.scatter(y_val_true, y_val_pred, s=20, alpha=0.8)

min_value = min(y_val_true.min(), y_val_pred.min())
max_value = max(y_val_true.max(), y_val_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], "k--", linewidth=1)

plt.xlabel("Observed")
plt.ylabel("Predicted")
plt.title("Observed vs predicted")
plt.grid(True)
plt.tight_layout()

scatter_path = RUN_DIR / "val_scatter.png"
plt.savefig(scatter_path, dpi=200)
plt.show()

print("散点图保存到:", scatter_path)

## 运行结果说明

这个 notebook 跑完后，`RUN_DIR` 目录中会有：

- `best_model.pth`：验证集 loss 最低的模型权重；
- `training_history.csv`：每个 epoch 的 train/val loss；
- `metrics.json`：RMSE、MAE、R2、相关系数等验证指标；
- `val_predictions.csv`：验证集逐日真实值、预测值和误差；
- `loss_curve.png`：训练曲线；
- `val_timeseries.png`：验证集时间序列对比图；
- `val_scatter.png`：真实值和预测值散点图。

注意：1985 单年样本很少，结果主要用于确认完整流程能跑通。要更接近论文效果，需要使用多年数据重新预处理和训练。